In [ ]:
from google.colab import drive

drive.mount("/content/drive")
BASE_DIR = "/content/drive/MyDrive/Nexlyra/05-10-2026-Ravi"

Mounted at /content/drive


In [ ]:
import os
import sys
import gc
import base64
import json
import psutil
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import librosa
from typing import Tuple, List, Dict, Any, Optional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Verify environment imports
print(f"PyTorch Version : {torch.__version__}")
print(f"Pandas Version  : {pd.__version__}")
print(f"NumPy Version   : {np.__version__}")
print(f"Librosa Version : {librosa.__version__}")

PyTorch Version : 2.11.0+cu130
Pandas Version  : 2.2.3
NumPy Version   : 2.1.3
Librosa Version : 0.11.0


In [ ]:
# Task 37: Reproducibility & Determinism Setup
def seed_everything(seed: int = 42) -> None:
    """Sets global random seeds for Python, NumPy, and PyTorch for full reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[INFO] Strict global seeds set to: {seed}")

seed_everything(42)

[INFO] Strict global seeds set to: 42


In [ ]:
# Task 38: Verify GPU Acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Compute Device: {device}")
if torch.cuda.is_available():
    print(f"[INFO] GPU Model: {torch.cuda.get_device_name(0)}")

[INFO] Compute Device: cuda
[INFO] GPU Model: Tesla T4


In [ ]:
# Task 03: Naive CSV Loading & Memory Spike Documentation
def log_ram_usage(step_description: str) -> None:
    """Logs current process RAM memory consumption."""
    process = psutil.Process(os.getpid())
    mem_mb = process.memory_info().rss / (1024 * 1024)
    print(f"[RAM LOG] {step_description}: {mem_mb:.2f} MB")

In [ ]:
DATA_PATH = BASE_DIR+"/nexlyra_hydro_cavitation_ml_1M_ravi.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"[ERROR] Required dataset file '{DATA_PATH}' not found in working directory."
    )

In [ ]:
# Task 04 & 05: Memory-Optimized Ingestion via PyArrow Types & Chunking
def load_and_preprocess_chunks(file_path: str, chunksize: int = 50000) -> pd.DataFrame:
    """
    Loads large hydrophone telemetry CSV using engine='c' with PyArrow backends,
    dropping non-predictive metadata ('Log_ID', 'Timestamp', 'Invoice_Amount_USD')
    and downcasting categorical and numeric types according to the true schema:
    ['Dam_Sector', 'Contractor', 'Turbine_Model', 'Billed_AntiCav_Hours',
     'IoT_Hydrophone_JSON', 'Cavitation_Triggered']
    """
    log_ram_usage("Before Chunked Processing")
    chunks = []

    for chunk in pd.read_csv(
        file_path,
        chunksize=chunksize,
        engine='c',
        dtype_backend='pyarrow'
    ):
        # Drop non-predictive financial/log columns to optimize RAM
        chunk = chunk.drop(
            columns=['Invoice_Amount_USD', 'Log_ID', 'Timestamp'],
            errors='ignore'
        )

        # Downcast categorical text columns
        chunk['Dam_Sector'] = chunk['Dam_Sector'].astype('category')
        chunk['Contractor'] = chunk['Contractor'].astype('category')
        chunk['Turbine_Model'] = chunk['Turbine_Model'].astype('category')

        # Downcast target and numerical metadata
        chunk['Cavitation_Triggered'] = chunk['Cavitation_Triggered'].astype(np.int8)
        chunk['Billed_AntiCav_Hours'] = chunk['Billed_AntiCav_Hours'].astype(np.float32)

        chunks.append(chunk)

    df = pd.concat(chunks, ignore_index=True)
    log_ram_usage("After Chunked Optimized Ingestion")
    return df

In [ ]:
df = load_and_preprocess_chunks(DATA_PATH)

[RAM LOG] Before Chunked Processing: 2232.98 MB
[RAM LOG] After Chunked Optimized Ingestion: 2404.77 MB


In [ ]:
def safe_json_loads(json_val: Any) -> Optional[Dict[str, Any]]:
    """Safely parses JSON strings or returns dictionary objects."""
    if pd.isna(json_val) or json_val is None:
        return None
    if isinstance(json_val, dict):
        return json_val
    try:
        return json.loads(str(json_val))
    except (json.JSONDecodeError, TypeError):
        return None

print("[INFO] Parsing outer IoT Hydrophone JSON payloads...")
parsed_dicts = df['IoT_Hydrophone_JSON'].apply(safe_json_loads)

[INFO] Parsing outer IoT Hydrophone JSON payloads...


In [ ]:
def extract_telemetry(payload: Optional[Dict[str, Any]]) -> Tuple[float, float, float, np.ndarray]:
    if payload is None:
        return np.nan, np.nan, np.nan, np.zeros(100, dtype=np.float32)

    press_kpa = float(payload.get('press_kpa', np.nan))
    vel_ms = float(payload.get('vel_ms', np.nan))
    temp_c = float(payload.get('temp_c', np.nan))

    sens_val = payload.get('sens_b85') or payload.get('spectrogram') or payload.get('sens') or ''

    spec_array = np.zeros(100, dtype=np.float32)

    try:
        if isinstance(sens_val, str) and len(sens_val.strip()) > 0:
            # Step 1: Base85 decode -> hex string
            hex_str = base64.b85decode(sens_val.strip().encode('utf-8')).decode('utf-8')

            # Step 2: Hex decode -> standard ASCII JSON string
            json_str = bytes.fromhex(hex_str).decode('utf-8')

            # Step 3: Parse JSON list of floats
            spec_list = json.loads(json_str)
            spec_array = np.array([float(v) for v in spec_list], dtype=np.float32)

        elif isinstance(sens_val, list):
            spec_array = np.array([float(v) for v in sens_val], dtype=np.float32)

        # Ensure uniform shape (100 elements)
        if len(spec_array) != 100:
            if len(spec_array) > 100:
                spec_array = spec_array[:100]
            else:
                spec_array = np.pad(spec_array, (0, 100 - len(spec_array)), mode='constant')

    except Exception:
        spec_array = np.zeros(100, dtype=np.float32)

    return press_kpa, vel_ms, temp_c, spec_array

In [ ]:
print("[INFO] Decoding Base85 spectrogram arrays and tabular metrics...")
extracted_data = [extract_telemetry(p) for p in parsed_dicts]

# Assign parsed scalar features back to DataFrame
df['press_kpa'] = [x[0] for x in extracted_data]
df['vel_ms'] = [x[1] for x in extracted_data]
df['temp_c'] = [x[2] for x in extracted_data]

# Extracted spectrogram arrays list for PyTorch conversion
spectrogram_arrays = [x[3] for x in extracted_data]

[INFO] Decoding Base85 spectrogram arrays and tabular metrics...


In [ ]:
# Task 12-15: Dynamic 1D Interpolation Collate Function
jagged_indices = [i for i, arr in enumerate(spectrogram_arrays) if len(arr) == 98]
print(f"[INFO] Identified {len(jagged_indices)} jagged telemetry tensors (98 elements).")

def hydrophone_collate_fn(
    batch: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Interpolates 1D audio tensors to 100 elements using F.interpolate,
    reshapes them to [1, 10, 10], and stacks the batch.
    """
    raw_specs, tabular_list, targets_list = zip(*batch)
    processed_specs = []

    for spec in raw_specs:
        curr_len = spec.numel()

        if curr_len == 100:
            resampled = spec.view(1, 10, 10)
        elif curr_len == 0:
            resampled = torch.zeros((1, 10, 10), dtype=torch.float32)
        else:
            spec_3d = spec.unsqueeze(0).unsqueeze(0)
            interpolated_3d = F.interpolate(
                spec_3d,
                size=100,
                mode='linear',
                align_corners=True
            )
            resampled = interpolated_3d.view(1, 10, 10)

        processed_specs.append(resampled)

    batch_spectrograms = torch.stack(processed_specs)  # [Batch_Size, 1, 10, 10]
    batch_tabular = torch.stack(tabular_list)            # [Batch_Size, 3]
    batch_targets = torch.stack(targets_list)            # [Batch_Size]

    return batch_spectrograms, batch_tabular, batch_targets


[INFO] Identified 0 jagged telemetry tensors (98 elements).


In [ ]:
raw_spectrogram_tensors = [torch.tensor(arr, dtype=torch.float32) for arr in spectrogram_arrays]

In [ ]:
# Task 16: Localized Grouped Imputation
df['press_kpa'] = df.groupby('Dam_Sector', observed=False)['press_kpa'].transform(
    lambda grp: grp.fillna(grp.rolling(window=5, min_periods=1).median())
)

df['press_kpa'] = df['press_kpa'].fillna(df['press_kpa'].median())
df['vel_ms'] = df['vel_ms'].fillna(df['vel_ms'].median())
df['temp_c'] = df['temp_c'].fillna(df['temp_c'].median())

In [ ]:
# Task 17 & 18: Stratified Dataset Splitting
positive_ratio = df['Cavitation_Triggered'].mean()
print(f"[INFO] Cavitation_Triggered Positive Class Ratio: {positive_ratio * 100:.2f}%")

indices = np.arange(len(df))
train_idx, val_test_idx = train_test_split(
    indices, test_size=0.30, stratify=df['Cavitation_Triggered'], random_state=42
)
val_idx, test_idx = train_test_split(
    val_test_idx, test_size=0.50, stratify=df['Cavitation_Triggered'].iloc[val_test_idx], random_state=42
)

print(f"[INFO] Data Split -> Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")


[INFO] Cavitation_Triggered Positive Class Ratio: 6.76%
[INFO] Data Split -> Train: 70000, Val: 15000, Test: 15000


In [ ]:
# Task 26-28: Tabular Normalization & Scaler Export
tabular_features = df[['press_kpa', 'vel_ms', 'temp_c']].values.astype(np.float32)

scaler = StandardScaler()
tabular_train_scaled = scaler.fit_transform(tabular_features[train_idx])
tabular_val_scaled = scaler.transform(tabular_features[val_idx])
tabular_test_scaled = scaler.transform(tabular_features[test_idx])

tabular_scaled_all = np.zeros_like(tabular_features)
tabular_scaled_all[train_idx] = tabular_train_scaled
tabular_scaled_all[val_idx] = tabular_val_scaled
tabular_scaled_all[test_idx] = tabular_test_scaled

scaler_path = BASE_DIR+"/hydrophone_scaler.joblib"
joblib.dump(scaler, scaler_path)
print(f"[INFO] Fitted StandardScaler saved to disk at: {scaler_path}")

[INFO] Fitted StandardScaler saved to disk at: /content/drive/MyDrive/Nexlyra/05-10-2026-Ravi/hydrophone_scaler.joblib


In [ ]:
# Task 19-21: PyTorch Hydrophone Dataset
class HydrophoneSpectrogramDataset(Dataset):
    def __init__(self, spectrograms: List[torch.Tensor], tabular: np.ndarray, targets: np.ndarray):
        self.spectrograms = spectrograms
        self.tabular = torch.tensor(tabular, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.targets)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        return self.spectrograms[idx], self.tabular[idx], self.targets[idx]

In [ ]:
targets_array = df['Cavitation_Triggered'].values

In [ ]:
train_dataset = HydrophoneSpectrogramDataset(
    [raw_spectrogram_tensors[i] for i in train_idx],
    tabular_scaled_all[train_idx],
    targets_array[train_idx]
)


In [ ]:
# Task 22-25: PyTorch DataLoader & Batch Verification

train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False,
    collate_fn=hydrophone_collate_fn
)

In [ ]:
spec_batch, tab_batch, target_batch = next(iter(train_loader))

In [ ]:
print(f"[VERIFICATION] Yielded Spectrogram Batch Shape: {spec_batch.shape}")
print(f"[VERIFICATION] Yielded Tabular Batch Shape    : {tab_batch.shape}")
print(f"[VERIFICATION] Yielded Target Batch Shape     : {target_batch.shape}")

[VERIFICATION] Yielded Spectrogram Batch Shape: torch.Size([512, 1, 10, 10])
[VERIFICATION] Yielded Tabular Batch Shape    : torch.Size([512, 3])
[VERIFICATION] Yielded Target Batch Shape     : torch.Size([512])


In [ ]:
# Task 29-33: Baseline Model & False Positive Breakdown by Contractor / Model
processed_spectrogram_tensors = []
for t in raw_spectrogram_tensors:
    if t.numel() == 100:
        processed_spectrogram_tensors.append(t.view(1, 10, 10))
    elif t.numel() == 0:
        processed_spectrogram_tensors.append(torch.zeros((1, 10, 10), dtype=torch.float32))
    else:
        resampled = F.interpolate(t.unsqueeze(0).unsqueeze(0), size=100, mode='linear', align_corners=True)
        processed_spectrogram_tensors.append(resampled.view(1, 10, 10))

all_max_vals = np.array([torch.max(t).item() for t in processed_spectrogram_tensors])
baseline_predictions = (all_max_vals > 4.0).astype(int)

prec = precision_score(targets_array, baseline_predictions, zero_division=0)
rec = recall_score(targets_array, baseline_predictions, zero_division=0)
f1 = f1_score(targets_array, baseline_predictions, zero_division=0)

print(f"\n--- Baseline Model Performance Metrics ---")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}")


--- Baseline Model Performance Metrics ---
Precision : 1.0000
Recall    : 0.9010
F1-Score  : 0.9479


In [ ]:
# Step 1: Reshape / Interpolate Spectrogram Tensors to Standard [1, 10, 10] Spatial Grid
processed_spectrogram_tensors = []
for t in raw_spectrogram_tensors:
    if t.numel() == 100:
        processed_spectrogram_tensors.append(t.view(1, 10, 10))
    elif t.numel() == 0:
        processed_spectrogram_tensors.append(torch.zeros((1, 10, 10), dtype=torch.float32))
    else:
        resampled = F.interpolate(
            t.unsqueeze(0).unsqueeze(0),
            size=100,
            mode='linear',
            align_corners=True
        )
        processed_spectrogram_tensors.append(resampled.view(1, 10, 10))

# Step 2: Compute Peak Amplitude & Naive Threshold (> 4.0) Predictions
all_max_vals = np.array([torch.max(t).item() for t in processed_spectrogram_tensors])
baseline_predictions = (all_max_vals > 4.0).astype(int)

# Extract ground truth target array
targets_array = df['Cavitation_Triggered'].values

# Step 3: Compute Baseline Classification Metrics
prec = precision_score(targets_array, baseline_predictions, zero_division=0)
rec = recall_score(targets_array, baseline_predictions, zero_division=0)
f1 = f1_score(targets_array, baseline_predictions, zero_division=0)

print("--- Baseline Model Performance Metrics ---")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-Score  : {f1:.4f}\n")

# Step 4: False Positive Analysis Across Subgroups
df['Baseline_Pred'] = baseline_predictions

# Filter False Positives (Predicted = 1, Actual = 0)
false_positives_df = df[(df['Baseline_Pred'] == 1) & (df['Cavitation_Triggered'] == 0)]

print("--- False Positives Breakdown by Contractor ---")
print(false_positives_df['Contractor'].value_counts())

print("\n--- False Positives Breakdown by Turbine Model ---")
print(false_positives_df['Turbine_Model'].value_counts())

--- Baseline Model Performance Metrics ---
Precision : 1.0000
Recall    : 0.9010
F1-Score  : 0.9479

--- False Positives Breakdown by Contractor ---
Contractor
Apex_Hydro_Ops            0
Global_Fluid_Dynamics     0
Hydro_Phantom_Turbines    0
Titan_Turbine_Tech        0
Name: count, dtype: int64

--- False Positives Breakdown by Turbine Model ---
Turbine_Model
Francis_Turbine    0
Kaplan_Turbine     0
Pelton_Wheel       0
Name: count, dtype: int64


In [ ]:
# Isolate False Negatives (Actual Cavitation = 1, Predicted Baseline = 0)
false_negatives_df = df[(df['Cavitation_Triggered'] == 1) & (df['Baseline_Pred'] == 0)]

print(f"Total Missed Cavitation Events: {len(false_negatives_df)}")
print("\n--- Physical Characteristics of Missed Events ---")
print(false_negatives_df[['press_kpa', 'vel_ms', 'temp_c', 'Billed_AntiCav_Hours']].describe())

Total Missed Cavitation Events: 669

--- Physical Characteristics of Missed Events ---
         press_kpa        vel_ms        temp_c  Billed_AntiCav_Hours
count   669.000000  6.690000e+02  6.690000e+02            669.000000
mean    745.765164  9.790000e+00  1.439000e+01             10.306328
std     181.564336  3.199835e-14  3.377603e-14              2.447769
min      16.700000  9.790000e+00  1.439000e+01              2.006418
25%     620.380000  9.790000e+00  1.439000e+01             10.869581
50%     756.560000  9.790000e+00  1.439000e+01             11.311187
75%     883.870000  9.790000e+00  1.439000e+01             11.668705
max    1175.710000  9.790000e+00  1.439000e+01             11.999541


1. High Hydrostatic Pressure SpikesThe mean pressure for missed events is $745.77 \text{ kPa}$ (peaking at $1175.71 \text{ kPa}$), which is significantly higher than normal operating baseline levels. Under high-pressure conditions, micro-cavitation bubbles implode rapidly before growing into large acoustic signatures, suppressing peak acoustic amplitudes below $4.0$ while still causing mechanical cavitation wear.

2. Constant Flow & Temperature BaselinesBoth fluid velocity (vel_ms $= 9.79 \text{ m/s}$) and fluid temperature (temp_c $= 14.39^\circ\text{C}$) have zero standard deviation ($\sigma \approx 0$). This indicates these missed events occur under steady, regulated operating regimes where noise is heavily damped.
3. High Billed Maintenance HoursBilled_AntiCav_Hours averages $10.31 \text{ hours}$, indicating these turbines were operating under heavy anti-cavitation maintenance routines, which dampen acoustic propagation even when cavitation physically occurs.

In [1]:
# Task 34: Save Processed PyTorch Tensors
processed_data_payload = {
    "train": {
        "spectrograms": torch.stack([processed_spectrogram_tensors[i] for i in train_idx]),
        "tabular": torch.tensor(tabular_scaled_all[train_idx], dtype=torch.float32),
        "targets": torch.tensor(targets_array[train_idx], dtype=torch.float32)
    },
    "val": {
        "spectrograms": torch.stack([processed_spectrogram_tensors[i] for i in val_idx]),
        "tabular": torch.tensor(tabular_scaled_all[val_idx], dtype=torch.float32),
        "targets": torch.tensor(targets_array[val_idx], dtype=torch.float32)
    },
    "test": {
        "spectrograms": torch.stack([processed_spectrogram_tensors[i] for i in test_idx]),
        "tabular": torch.tensor(tabular_scaled_all[test_idx], dtype=torch.float32),
        "targets": torch.tensor(targets_array[test_idx], dtype=torch.float32)
    }
}

NameError: name 'torch' is not defined

In [ ]:
tensor_output_path = BASE_DIR+"/hydrophone_processed_tensors.pt"
torch.save(processed_data_payload, tensor_output_path)
print(f"[INFO] Processed PyTorch Train/Val/Test tensors saved to disk at: {tensor_output_path}")

[INFO] Processed PyTorch Train/Val/Test tensors saved to disk at: /content/drive/MyDrive/Nexlyra/05-10-2026-Ravi/hydrophone_processed_tensors.pt


In [ ]:
log_ram_usage("Before Memory Cleanup")

# Step 1: Delete large raw DataFrames, Series, and intermediate extraction payloads
del df
del processed_data_payload
del raw_spectrogram_tensors
del extracted_data
del parsed_dicts

# Step 2: Force Python Garbage Collection
gc.collect()

# Step 3: Clear PyTorch CUDA / MPS GPU Cache (if training on GPU acceleration)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    torch.mps.empty_cache()

log_ram_usage("After Memory Cleanup")

[RAM LOG] Before Memory Cleanup: 2098.27 MB
[RAM LOG] After Memory Cleanup: 1783.73 MB
